# Schönberg – Universal-Edition: Themen und Beziehungen

Katharina Bleier · Workshop „Netzwerken in Wien um 1900“, November 2026

**Frage:** Die Metadaten einer Briefedition zeigen, *wer mit wem* korrespondiert. Kann Topic Modeling sichtbar machen,
*worüber* – und *wie* – in diesen Beziehungen kommuniziert wird?

**Daten:** Arnold Schönberg: Briefwechsel mit den Verlagen Universal-Edition und Dreililien
(Hg. Katharina Bleier, Therese Muxeneder; mdw/Arnold Schönberg Center), TEI-Datenpaket
[Schoenberg-Data](https://github.com/Schoenberg-UE/Schoenberg-Data) v2.5, CC BY-SA 4.0.

**Ablauf:** TEI → Sätze → Topics → Themengruppen → Profile der Korrespondenzbeziehungen

| Schritt | Entscheidung |
|---|---|
| Textgrundlage | nur Briefe; Brieftext aus `<body>` ohne Stellenkommentar, Briefkopf, Formeln, Gestrichenes; Korrekturen statt Fehler; getrennte Wörter verbunden |
| Einheit | Satz (mindestens 4 Wörter) |
| Modell | BERTopic: Sätze werden nach ihrer Bedeutung gruppiert (Sprachmodell `paraphrase-multilingual-mpnet-base-v2`) |
| Topic-Wörter | nur Substantive und Eigennamen, ohne ausgezeichnete Personen-, Orts- und Organisationsnamen, Schreibvarianten vereinheitlicht, lemmatisiert |
| Interpretation | Topics benannt und zu 16 Gruppen zusammengefasst: Themen und Sprechhandlungen |
| Überlieferung | erschlossene Briefe zählen als Kontakt, haben aber kein Themenprofil |

In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from asue_topics import extract, model, aggregate as ag
from asue_topics.normalize import NormalizeConfig

TEI_DIR = ROOT.parent / "Schoenberg-Data" / "data"   # Datenrepo neben diesem Projekt
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

## 1. Briefe aus TEI extrahieren

In [ ]:
xcfg = extract.ExtractConfig(unit="sentence")
extract.run(TEI_DIR, OUT, xcfg)
letters_all = pd.read_csv(OUT / "letters.csv")
letters_all["status"].value_counts()

## 2. Text für das Modell aufbereiten
Das Sprachmodell sieht den **Originalsatz** (`text`). Für die Wörter, die ein Topic beschreiben, wird eine zweite Fassung
gebildet (`text_repr`): ohne Namen, Schreibvarianten vereinheitlicht (Ersetzungsliste: `output/normalization_map.csv`),
lemmatisiert, nur Substantive.

In [ ]:
mcfg = model.ModelConfig(
    min_words=4,
    embedding_model="paraphrase-multilingual-mpnet-base-v2",
    min_cluster_size=15,
    min_samples=5,
    nr_topics=30,
    merge_topics=[[4, 11]],
    normalize=NormalizeConfig(orthography=True, lemmatize=True, spacy_model="de_core_news_md"),
)
units = model.load_units(OUT, mcfg)
units[["text", "text_repr"]].sample(5, random_state=1)

## 3. Topic Modeling
Beim ersten Lauf werden die Embeddings berechnet (einige Minuten) und in `output/cache/` gespeichert.

In [ ]:
topic_model, res, info, emb = model.fit_bertopic(units, mcfg, OUT / "cache")
model.export(res, info, OUT, "bertopic")
info[["topic", "count", "top_words"]]

## 4. Topics benennen und gruppieren
Benennung nach Topic-Wörtern **und** Beispielsätzen (`output/bertopic_topic_info.csv`).
Die Nummern gelten für die Einstellungen oben; bei geänderten Parametern neu prüfen.

In [ ]:
LABELS = {
    0: "Vertrag/Vertragsverhältnis", 1: "Termine/Urgenzen", 2: "Beilagen/Sendungen", 3: "Aufführungen",
    4: "Orchesterwerke/Kammermusik", 5: "Zahlungen/Konto", 6: "Position/Stellung", 7: "Vermittlung/Dritte",
    8: "Publikation/Copyright", 9: "Befindlichkeit", 10: "Ablehnung", 11: "Werkstellen/Korrekturverweise",
    12: "Geldbeträge/Forderungen", 13: "Ausbleibende Antworten", 14: "Notensatz/Stich", 15: "Pelleas/Pierrot",
    16: "Druckumfang/Seiten", 17: "Listen/Versandangaben", 18: "Arbeit/Zeitnot", 19: "Eindrücke/Erfolg",
    20: "Schwierigkeiten", 21: "Ungewissheit", 22: "Preise/Auflage", 23: "Abrechnungszeiträume/Daten",
    24: "Paris", 25: "Adressen", 26: "Konflikt um Hugo Winter", 27: "Englisch/Übersetzung",
}
GRUPPEN = {
    "Vertrag/Vertragsverhältnis": [0],
    "Publikation/Copyright": [8],
    "Geld/Abrechnung": [5, 12, 22, 23],
    "Werke": [4, 15],
    "Herstellung/Notensatz": [11, 14, 16],
    "Aufführungen": [3, 19, 24],
    "Beilagen/Sendungen": [2, 17, 25],
    "Englisch/Übersetzung": [27],
    "Position/Stellung": [6],
    "Urgenzen/Termine": [1, 13],
    "Vermittlung/Dritte": [7],
    "Ablehnung": [10],
    "Schwierigkeiten/Ungewissheit": [20, 21],
    "Befindlichkeit": [9],
    "Arbeit/Zeitnot": [18],
    "Konflikt um Hugo Winter": [26],
}
THEMEN = ["Vertrag/Vertragsverhältnis", "Publikation/Copyright", "Geld/Abrechnung", "Werke",
          "Herstellung/Notensatz", "Aufführungen", "Beilagen/Sendungen", "Englisch/Übersetzung",
          "Position/Stellung"]
SPRECHHANDLUNGEN = [g for g in GRUPPEN if g not in THEMEN]
KATEGORIE = {**{g: "Thema" for g in THEMEN}, **{g: "Sprechhandlung" for g in SPRECHHANDLUNGEN}}
TOPIC_ZU_GRUPPE = {t: g for g, ts in GRUPPEN.items() for t in ts}

# Labels in die Datei schreiben (für die Dokumentation)
lab = info[["topic", "count", "top_words"]].copy()
lab["label"] = lab["topic"].map(LABELS).fillna("Outlier")
lab["gruppe"] = lab["topic"].map(TOPIC_ZU_GRUPPE).fillna("")
lab.to_csv(OUT / "bertopic_topic_labels.csv", index=False)

res["gruppe"] = res["topic"].map(TOPIC_ZU_GRUPPE)
fehlend = sorted(set(res.loc[res.topic != -1, "topic"]) - set(TOPIC_ZU_GRUPPE))
print("Topics ohne Gruppe:", fehlend or "keine")
lab[["topic", "count", "label", "gruppe"]]

## 5. Von Sätzen zu Beziehungen
Anteil jeder Gruppe an den Sätzen eines Briefs (ohne Outlier) → Mittelwert je Korrespondenzpartner:in.
Korrespondenzpartner:in = unterzeichnende Person aus `correspDesc` (sonst die Körperschaft).

In [ ]:
letters = ag.add_period(ag.add_partner(letters_all, xcfg.schoenberg_name_pattern))
lt = ag.letter_topics(res)                                        # Brief × Topic
lt = lt.T.groupby(lt.columns.map(TOPIC_ZU_GRUPPE)).sum().T         # Brief × Gruppe
labels = {g: g for g in lt.columns}

prof_partner = ag.profile(lt, letters, "partner", min_letters=15)
prof_partner[["n_letters", "n_erschlossen"]]

## Abbildung 1: Worüber schreibt Schönberg mit wem?
Je Korrespondenzpartner:in die fünf häufigsten Gruppen (Anteil der Sätze); blau = Themen, orange = Sprechhandlungen.

In [ ]:
ag.plot_partner_profiles(
    prof_partner, labels, OUT / "abb1_themenprofile.png",
    partners=["Hertzka, Emil", "Kalmus, Alfred", "Marschalk, Max",
              "Schlee, Alfred", "Stein, Erwin", "Winter, Hugo"],
    categories=KATEGORIE, top_k=5,
    title="Worüber schreibt Schönberg mit wem?",
)

## Abbildung 2: Themen im Zeitverlauf
Gleitender 3-Jahres-Mittelwert; graue Balken = Briefe pro Jahr. In Jahren mit wenigen Briefen schwanken die Linien stark.

In [ ]:
ag.plot_topics_over_time(
    lt, letters, labels, OUT / "abb2_zeitverlauf.png",
    topics=["Geld/Abrechnung", "Aufführungen", "Urgenzen/Termine"],
    markers={1933: "Emigration"},
    year_range=(1902, 1938),
    title="Themen der Korrespondenz im Zeitverlauf",
)

## 6. Close Reading
Trägt die Zuordnung? Sätze einer Gruppe in einer Beziehung ansehen.

In [ ]:
GRUPPE, PARTNER = "Urgenzen/Termine", "Winter, Hugo"
ids = letters.loc[letters["partner"] == PARTNER, "letter_id"]
sel = res[(res["gruppe"] == GRUPPE) & (res["letter_id"].isin(ids))]
print(f"{GRUPPE} | {PARTNER} | {len(sel)} Sätze")
sel[["letter_id", "text"]].head(15)

## 7. Kennzahlen

In [ ]:
n_out = (res["topic"] == -1).mean()
print(f"Briefe: {len(letters)} (überliefert {sum(letters.status == 'überliefert')}, "
      f"erschlossen {sum(letters.status == 'erschlossen')})")
print(f"Sätze im Modell: {len(units)}")
print(f"Topics: {(info.topic != -1).sum()} → {len(GRUPPEN)} Gruppen")
print(f"Sätze ohne Topic (Outlier): {n_out:.0%}")